# BitcoinGraphGuard — Phase 2: XGBoost Baseline

**Phase:** 2 of 9 · Classical supervised baseline
**Environment:** Google Colab / Kaggle. This notebook is **never executed on the laptop**.
**Input:** the real Elliptic++ transactions in `Og data/`, using the protocol and findings from
`docs/EDA.md` and `notebooks/01_eda.ipynb`.

Model progression for the project:

```text
Simple baseline -> Logistic Regression -> XGBoost -> GraphSAGE -> RGCN / HGT
```

This notebook implements the first three steps. It answers one question: **how much predictive
signal exists in the transaction features alone, under a leakage-safe temporal protocol?** That
number is the bar GraphSAGE and the heterogeneous GNNs have to clear in later phases.

What this notebook deliberately does **not** do: build a graph, use any graph statistic as a
feature, touch the test period during fitting or model selection, or treat unknown labels as licit.

**To run:** `Runtime -> Run all`. Everything is written to `<Drive>/Projects/Bitcoin Graph/xgboost/`.

## 1. Setup

One config cell. Seeds, time-step constants, path resolution and the small helpers used throughout.
No personal paths are hardcoded — the data directory is discovered from the environment.

In [ ]:
import gc
import json
import os
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
CHUNK = 50_000           # rows per streamed read, kept for parity with notebook 01
N_STEPS = 49             # Elliptic++ covers 49 contiguous time steps

# Temporal protocol (see section 3). FIT is the early-stopping/threshold window carved out of
# the training period; VAL is historical and never overlaps the test period.
FIT_MIN, FIT_MAX = 1, 24
VAL_MIN, VAL_MAX = 25, 34
TRAIN_MIN, TRAIN_MAX = 1, 34
TEST_MIN, TEST_MAX = 35, 49
TEST_EARLY_MAX = 42      # 35-42 sub-window
DRIFT_MIN = 43           # 43-49 secondary recent-drift window

LABEL_NAMES = {1: "illicit", 2: "licit", 3: "unknown"}

np.random.seed(SEED)
sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

print(f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"xgboost {xgb.__version__}")

In [ ]:
FILES = {
    "txs_features": "txs_features.csv",
    "txs_classes": "txs_classes.csv",
    "addr_tx": "AddrTx_edgelist.csv",
    "tx_addr": "TxAddr_edgelist.csv",
}

# One Drive folder holds the raw dataset and every notebook's outputs. Change this single line
# (or set BITCOINGUARD_DRIVE_ROOT) if your layout differs.
DRIVE_ROOT = Path(os.environ.get("BITCOINGUARD_DRIVE_ROOT",
                                 "/content/drive/MyDrive/Projects/Bitcoin Graph"))


def resolve_data_dir() -> Path:
    """Find the Elliptic++ CSVs in Colab, Kaggle or a local checkout."""
    override = os.environ.get("ELLIPTIC_DATA_DIR")
    candidates = [
        Path(override) if override else None,
        DRIVE_ROOT / "Dataset",
        Path("/content/drive/MyDrive/elliptic-plus-plus"),
        Path("/content/elliptic-plus-plus"),
        Path("/kaggle/input/elliptic-plus-plus"),
        Path("/kaggle/input/elliptic"),
        Path.cwd() / "Og data",
    ]
    for candidate in candidates:
        if candidate and candidate.is_dir() and (candidate / FILES["txs_features"]).exists():
            return candidate
    raise FileNotFoundError("Elliptic++ not found; set ELLIPTIC_DATA_DIR to the folder with the CSVs.")


def read_header(path: Path) -> list:
    """Column names of a CSV without reading any data rows."""
    return pd.read_csv(path, nrows=0).columns.tolist()


DATA_DIR = resolve_data_dir()
OUT_DIR = DRIVE_ROOT / "xgboost"
FIG_DIR = OUT_DIR / "figures"
for directory in (OUT_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CHECKS: list = []


def check(name: str, value, expected=None, note: str = "") -> None:
    """Record a sanity check so section 13 can report exactly what was verified."""
    ok = expected is None or value == expected
    CHECKS.append({"check": name, "value": value, "expected": expected, "ok": bool(ok), "note": note})
    flag = "ok" if ok else "FAIL"
    print(f"[{flag}] {name}: {value}" + (f" (expected {expected})" if expected is not None else ""))


def save_table(name: str, frame: pd.DataFrame) -> None:
    frame.to_csv(OUT_DIR / name, index=False)


def save_json(name: str, payload: dict) -> None:
    (OUT_DIR / name).write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def save_fig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=130, bbox_inches="tight")
    plt.show()


print(f"data : {DATA_DIR}")
print(f"out  : {OUT_DIR}")

## 2. Load Data

The raw dataset is opened read-only and never modified.

Only two files are needed as features: `txs_features.csv` (165 model columns are read; the 17
domain columns are skipped at parse time) and `txs_classes.csv` for labels. The two address edge
lists are read for one derived column only — whether a transaction has at least one address link.

Rows are read once, in a single pass, with `float32` features. On Colab this is ~135 MB.

In [ ]:
txs_columns = read_header(DATA_DIR / FILES["txs_features"])
all_feature_columns = [c for c in txs_columns if c not in ("txId", "Time step")]
domain_columns = [c for c in all_feature_columns
                  if not c.startswith(("Local_feature", "Aggregate_feature"))]
model_features = [c for c in all_feature_columns if c not in domain_columns]

print(f"{len(all_feature_columns)} feature columns = {len(domain_columns)} domain + "
      f"{len(model_features)} Local/Aggregate")
print("domain block (excluded from the primary model):")
print(domain_columns)

check("features: total columns", len(all_feature_columns), 182)
check("features: domain columns", len(domain_columns), 17)
check("features: model columns before has_addresses", len(model_features), 165)
check("features: domain block is disjoint from model features",
      set(domain_columns).isdisjoint(model_features), True)

txs_classes = pd.read_csv(DATA_DIR / FILES["txs_classes"], dtype={"txId": str})
check("labels: transaction rows", len(txs_classes), 203_769)
check("labels: values are a subset of 1/2/3",
      set(txs_classes["class"].unique()) <= {1, 2, 3}, True)

In [ ]:
started = time.time()
txs = pd.read_csv(
    DATA_DIR / FILES["txs_features"],
    usecols=["txId", "Time step"] + model_features,
    dtype={"txId": str, **{c: "float32" for c in model_features}},
)
txs["class"] = txs["txId"].map(txs_classes.set_index("txId")["class"])

print(f"read {len(txs):,} transaction rows in {time.time() - started:.0f}s | "
      f"{txs.memory_usage(deep=True).sum() / 1e6:.0f} MB")

check("transactions: rows", len(txs), 203_769)
check("transactions: labels present for every row", int(txs["class"].isna().sum()), 0)
check("transactions: time step range", (int(txs["Time step"].min()), int(txs["Time step"].max())), (1, 49))
check("transactions: missing values in the 165 model features",
      int(txs[model_features].isna().to_numpy().sum()), 0,
      note="the domain block holds all missingness and it is excluded")

label_counts = {LABEL_NAMES[int(k)]: int(v) for k, v in txs["class"].value_counts().items()}
print(label_counts)
check("transactions: illicit count", label_counts.get("illicit"), 4_545)
check("transactions: licit count", label_counts.get("licit"), 42_019)
check("transactions: unknown count", label_counts.get("unknown"), 157_205)

In [ ]:
def transaction_ids_with_address_links() -> set:
    """Transaction ids appearing in AddrTx or TxAddr. Read-only, two columns only."""
    ids: set = set()
    for key in ("addr_tx", "tx_addr"):
        frame = pd.read_csv(DATA_DIR / FILES[key], usecols=["txId"], dtype={"txId": str})
        ids |= set(frame["txId"])
        del frame
        gc.collect()
    return ids


linked_ids = transaction_ids_with_address_links()
txs["has_addresses"] = txs["txId"].isin(linked_ids).astype("int8")
model_features = model_features + ["has_addresses"]

print(f"transactions with an address link : {int(txs['has_addresses'].sum()):,}")
print(f"transactions without one          : {int((txs['has_addresses'] == 0).sum()):,}")
check("has_addresses: address-less transactions", int((txs["has_addresses"] == 0).sum()), 965,
      note="matches the blank-domain rows found in Phase 1/EDA")

del linked_ids
gc.collect()

## 3. Define Temporal Protocol

The EDA adopted **train 1–34 / test 35–49** as the primary benchmark, with `43–49` kept as a
secondary recent-drift window (its illicit share of labeled transactions is 2.53% against 11.58% in
1–34, so it measures generalization under a low-prevalence regime, not the headline result).

Validation is carved out of the **historical** period, never out of the future:

| Window | Steps | Role |
| :--- | :--- | :--- |
| fit | 1–24 | early stopping and threshold selection |
| validation | 25–34 | historical validation |
| train | 1–34 | final model refit |
| test | 35–49 | final evaluation, untouched until section 9 |
| test_early | 35–42 | temporal sub-window |
| test_drift | 43–49 | secondary recent-drift window |

**Why `fit` stops at 24.** The protocol names 25–34 as validation, and 25–34 sits *inside* the
training period. Training on 1–34 while early-stopping on 25–34 would let the model stop on rows it
had already seen, and any threshold chosen there would be optimistic. So the model is first fit on
1–24 and validated on 25–34 to pick the number of trees and the operating threshold; it is then
refit on the full 1–34 and the frozen threshold is applied to the test period. The test period is
read by nothing but the final evaluation.

Sub-windows `35–42` and `43–49` partition the test period exactly, so their results are directly
comparable to `35–49`.

In [ ]:
step = txs["Time step"].astype("int16")
is_labeled = txs["class"].isin([1, 2])

masks = {
    "fit":        (step >= FIT_MIN) & (step <= FIT_MAX) & is_labeled,
    "validation": (step >= VAL_MIN) & (step <= VAL_MAX) & is_labeled,
    "train":      (step >= TRAIN_MIN) & (step <= TRAIN_MAX) & is_labeled,
    "test":       (step >= TEST_MIN) & (step <= TEST_MAX) & is_labeled,
    "test_early": (step >= TEST_MIN) & (step <= TEST_EARLY_MAX) & is_labeled,
    "test_drift": (step >= DRIFT_MIN) & (step <= TEST_MAX) & is_labeled,
}
masks = {name: mask.to_numpy() for name, mask in masks.items()}

window_rows = []
for name, mask in masks.items():
    labels = txs.loc[mask, "class"]
    positives = int((labels == 1).sum())
    labeled = int((labels == 2).sum())
    window_rows.append({
        "window": name,
        "steps": {"fit": "1-24", "validation": "25-34", "train": "1-34", "test": "35-49",
                  "test_early": "35-42", "test_drift": "43-49"}[name],
        "labeled": positives + labeled,
        "illicit": positives,
        "licit": labeled,
        "illicit_pct_of_labeled": 100 * positives / max(1, positives + labeled),
    })
window_table = pd.DataFrame(window_rows)
save_table("split_windows.csv", window_table)
display(window_table)

check("protocol: no train/test overlap", bool((masks["train"] & masks["test"]).any()), False)
check("protocol: sub-windows partition the test period",
      int(masks["test_early"].sum() + masks["test_drift"].sum()), int(masks["test"].sum()))
check("protocol: fit is a subset of train", bool(np.all(~masks["fit"] | masks["train"])), True)
check("protocol: validation is historical (<= 34)",
      bool((txs.loc[masks["validation"], "Time step"] <= 34).all()), True)

# Cross-check against the Phase 1 EDA artifacts (eda/split_windows.csv is the source of truth).
check("protocol: train labeled rows", int(masks["train"].sum()), 29_894)
check("protocol: train illicit rows", int((txs.loc[masks["train"], "class"] == 1).sum()), 3_462)
check("protocol: validation labeled rows", int(masks["validation"].sum()), 6_288)
check("protocol: validation illicit rows", int((txs.loc[masks["validation"], "class"] == 1).sum()), 1_243)
check("protocol: test labeled rows", int(masks["test"].sum()), 16_670)
check("protocol: test illicit rows", int((txs.loc[masks["test"], "class"] == 1).sum()), 1_083)
check("protocol: test_drift illicit rows", int((txs.loc[masks["test_drift"], "class"] == 1).sum()), 169)

## 4. Build Leakage-Safe Dataset

**Features.** The 165 anonymous `Local_feature_*` (93) and `Aggregate_feature_*` (72) columns, plus
`has_addresses`. Nothing else.

**The 17 domain columns are excluded.** The EDA showed every one of them has a `Local_feature_*`
counterpart at `|r| ≈ 0.98 – 1.00` (`eda/txs_domain_redundancy.csv`) — they are re-expressions, not
independent evidence. They are also the *only* columns with missing values: 17 × 965 blank cells,
exactly the 965 address-less transactions. Keeping them would force a masking decision for columns
that add nothing the `Local_feature_*` block does not already carry.

**`has_addresses` is a structural fact about the transaction itself**, not a graph statistic: a
transaction's inputs and outputs are part of the transaction, so the flag is known at prediction
time. It is added so the model can treat the 965 address-less transactions as a distinct population
instead of silently absorbing them as zeros.

**Population.** Class 1 (illicit) and class 2 (licit) only. Class 3 (unknown) is never a training or
evaluation example.

**No graph statistics.** Degrees, component sizes and hub ranks computed in the EDA cover the whole
transductive graph including future steps; they are excluded by construction here (section 5).

In [ ]:
inputs = {
    "fit":       (txs.loc[masks["fit"], model_features],       txs.loc[masks["fit"], "class"]),
    "validation": (txs.loc[masks["validation"], model_features], txs.loc[masks["validation"], "class"]),
    "train":     (txs.loc[masks["train"], model_features],     txs.loc[masks["train"], "class"]),
    "test":      (txs.loc[masks["test"], model_features],      txs.loc[masks["test"], "class"]),
}
X = {name: frame.astype("float32").reset_index(drop=True) for name, (frame, _) in inputs.items()}
y = {name: (labels == 1).astype("int8").reset_index(drop=True) for name, (_, labels) in inputs.items()}
test_step = txs.loc[masks["test"], "Time step"].astype("int16").reset_index(drop=True)
test_id = txs.loc[masks["test"], "txId"].reset_index(drop=True)

for name in ("fit", "validation", "train", "test"):
    positives = int(y[name].sum())
    print(f"{name:<11} rows {len(X[name]):>7,} | illicit {positives:>6,} "
          f"({100 * y[name].mean():.2f}%) | scale_pos_weight {(len(y[name]) - positives) / positives:.2f}")

check("dataset: fit has both classes", sorted(y["fit"].unique().tolist()), [0, 1])
check("dataset: unknown labels excluded", int(txs.loc[masks["train"] | masks["test"], "class"].isin([3]).sum()), 0)
check("dataset: feature matrix is float32", sorted({str(d) for d in X["train"].dtypes}), ["float32"])
check("dataset: every model feature is used", len(model_features), X["train"].shape[1])

save_json("feature_list.json", {"model_features": model_features,
                                "excluded_domain_columns": domain_columns,
                                "n_features": len(model_features)})

## 5. Leakage Audit

Every feature in the matrix is checked against the same six questions. The audit is a table, not a
paragraph, so a reviewer can see the answer for each group.

| # | Question |
| :--- | :--- |
| 1 | Was it available at prediction time? |
| 2 | Does it use information from future time steps? |
| 3 | Does it use labels? |
| 4 | Does it use test data? |
| 5 | Was it calculated on the complete graph, including future edges? |
| 6 | Does it aggregate across future observations? |

Verdicts used: **SAFE**, **EXCLUDED-LEAK**, **EXCLUDED-REDUNDANT**, **EXCLUDED-ID**,
**ASSUMED-SAFE** (safe by the dataset's construction, which we cannot fully re-derive from the raw
files — stated rather than hidden).

### Where each exclusion is enforced

* The 17 domain columns never leave the CSV (dropped in the `usecols` of the read).
* `txId` and `Time step` are never added to `model_features` — `Time step` defines the split, so
  using it as a feature would hand the model the experiment design.
* No degree, component, hub or neighbourhood-aggregate column is constructed anywhere in this
  notebook. The address edge lists are read for membership only, which is why `has_addresses` is
  safe and a degree would not be.
* No target encoding, no label-derived feature, no imputation statistic computed outside the fit
  window.

### Checks that this holds

Beyond the table, three things are asserted programmatically: the excluded columns are absent from
the feature matrix, the fitted preprocessing statistics (median, mean, scale) come only from the fit
window, and no single feature separates the labels implausibly well on the fit window (a single
feature with near-perfect AUC would be the signature of a leaked column).

In [ ]:
audit_rows = [
    {"feature_group": "Local_feature_* (93)", "available_at_prediction_time": "yes",
     "uses_future_steps": "no", "uses_labels": "no", "uses_test_data": "no",
     "uses_full_graph": "no", "future_aggregates": "no", "verdict": "ASSUMED-SAFE",
     "reason": "Bundle features published with Elliptic++; fixed at the transaction's own step."},
    {"feature_group": "Aggregate_feature_* (72)", "available_at_prediction_time": "yes",
     "uses_future_steps": "no", "uses_labels": "no", "uses_test_data": "no",
     "uses_full_graph": "no", "future_aggregates": "no", "verdict": "ASSUMED-SAFE",
     "reason": "Same provenance as Local. We do not re-derive them, so step-boundedness is assumed, not proven."},
    {"feature_group": "has_addresses", "available_at_prediction_time": "yes",
     "uses_future_steps": "no", "uses_labels": "no", "uses_test_data": "no",
     "uses_full_graph": "no", "future_aggregates": "no", "verdict": "SAFE",
     "reason": "Membership in the transaction's own address edge lists; part of the transaction."},
    {"feature_group": "domain block (17)", "available_at_prediction_time": "yes",
     "uses_future_steps": "no", "uses_labels": "no", "uses_test_data": "no",
     "uses_full_graph": "no", "future_aggregates": "no", "verdict": "EXCLUDED-REDUNDANT",
     "reason": "|r| ~ 0.98-1.00 with Local_feature_*; sole source of the 16,405 blanks."},
    {"feature_group": "txId", "available_at_prediction_time": "yes",
     "uses_future_steps": "no", "uses_labels": "no", "uses_test_data": "no",
     "uses_full_graph": "no", "future_aggregates": "no", "verdict": "EXCLUDED-ID",
     "reason": "Deterministic identifier; carries ordering information, no signal."},
    {"feature_group": "Time step", "available_at_prediction_time": "yes",
     "uses_future_steps": "no", "uses_labels": "no", "uses_test_data": "no",
     "uses_full_graph": "no", "future_aggregates": "no", "verdict": "EXCLUDED-ID",
     "reason": "Defines the split; a feature would let the model read the experiment design."},
    {"feature_group": "degree / component / hub statistics", "available_at_prediction_time": "no",
     "uses_future_steps": "yes", "uses_labels": "no", "uses_test_data": "yes",
     "uses_full_graph": "yes", "future_aggregates": "yes", "verdict": "EXCLUDED-LEAK",
     "reason": "EDA computed them over the transductive graph; they encode future edges and test nodes."},
    {"feature_group": "neighbourhood / wallet aggregates", "available_at_prediction_time": "no",
     "uses_future_steps": "yes", "uses_labels": "no", "uses_test_data": "yes",
     "uses_full_graph": "yes", "future_aggregates": "yes", "verdict": "EXCLUDED-LEAK",
     "reason": "Not used in Phase 2. Any future graph feature must be step-bounded to be admissible."},
    {"feature_group": "label-derived / target encoding", "available_at_prediction_time": "no",
     "uses_future_steps": "n/a", "uses_labels": "yes", "uses_test_data": "n/a",
     "uses_full_graph": "n/a", "future_aggregates": "n/a", "verdict": "EXCLUDED-LEAK",
     "reason": "None are constructed anywhere in this notebook."},
]
audit_table = pd.DataFrame(audit_rows)
save_table("leakage_audit.csv", audit_table)
display(audit_table[["feature_group", "uses_future_steps", "uses_labels", "uses_test_data",
                     "uses_full_graph", "verdict"]])

check("leakage: no domain column in the feature matrix",
      set(domain_columns).isdisjoint(set(X["train"].columns)), True)
check("leakage: no identifier column in the feature matrix",
      set(X["train"].columns).isdisjoint({"txId", "Time step", "address"}), True)
check("leakage: no feature constructed from labels",
      [c for c in X["train"].columns if "label" in c.lower() or "class" in c.lower()], [])

# A leaked column usually looks like a near-perfect single-feature classifier. Rank every feature
# by its AUC on the fit window; a value close to 1.0 means the audit above missed something.
fit_auc = []
for column in X["fit"].columns:
    scores = X["fit"][column].to_numpy(dtype="float64")
    fit_auc.append(roc_auc_score(y["fit"], scores) if 0 < y["fit"].sum() < len(y["fit"]) else 0.5)
fit_auc = np.array(fit_auc)
strongest = float(np.nanmax(np.abs(fit_auc - 0.5)) + 0.5)
print(f"strongest single-feature AUC on the fit window: {strongest:.3f} "
      f"({X['fit'].columns[int(np.nanargmax(np.abs(fit_auc - 0.5)))]})")
check("leakage: no single feature is a near-perfect discriminator", bool(strongest <= 0.95), True,
      note="EDA's strongest |r| with the label is 0.26, so an AUC above 0.95 would be a red flag")
check("leakage: preprocessing statistics will be fit on the fit window only", True, None,
      note="enforced in sections 7 and 8 by passing only X_fit to .fit()")

## 6. Simple Baseline

Before any model, establish what "no model" scores. Two trivial references:

* **Prevalence baseline** — score every transaction with the training-period illicit rate. A
  constant score has ROC-AUC 0.5 and a PR-AUC equal to the illicit prevalence **of the window it
  is evaluated on** — 6.50% on the test period, not the training rate it was built from. That
  distinction matters: the constant baseline gets *harder* to beat as prevalence falls.
* **Flag-everything detector** — the other extreme: recall 1.0, precision equal to the prevalence.

Neither is a real detector. They bracket the range that any model has to beat, and they are
evaluated on the test period with the same functions used later. The test period's illicit rate is
6.50% of labeled transactions, well below the training rate, so the precision of a flag-everything
detector collapses accordingly.

This is deliberately not over-engineered.

In [ ]:
baseline_prevalence_fit = float(y["fit"].mean())
baseline_prevalence_train = float(y["train"].mean())
test_prevalence = float(y["test"].mean())

constant_scores = np.full(len(y["test"]), baseline_prevalence_train)
baseline_constant_ap = float(average_precision_score(y["test"], constant_scores))
test_illicit = int(y["test"].sum())
test_licit = int(len(y["test"]) - test_illicit)

baseline_rows = [
    {"model": "Prevalence (constant score)",
     "pr_auc": baseline_constant_ap, "roc_auc": 0.5,
     "precision": test_prevalence, "recall": 1.0,
     "f1": 2 * test_prevalence / (1 + test_prevalence),
     "predicted_positives": len(y["test"]), "tp": test_illicit, "fp": test_licit,
     "tn": 0, "fn": 0, "threshold": baseline_prevalence_train,
     "note": "flag everything: recall 1, precision = prevalence"},
    {"model": "Majority class (all licit)",
     "pr_auc": baseline_constant_ap, "roc_auc": 0.5,
     "precision": 0.0, "recall": 0.0, "f1": 0.0,
     "predicted_positives": 0, "tp": 0, "fp": 0, "tn": test_licit, "fn": test_illicit,
     "threshold": 1.01,
     "note": "flag nothing: threshold-independent metrics are identical for any constant score"},
]
baseline_table = pd.DataFrame(baseline_rows)
display(baseline_table)

print(f"fit-window prevalence       : {baseline_prevalence_fit:.4f}")
print(f"train-window prevalence     : {baseline_prevalence_train:.4f}")
print(f"test-window prevalence      : {test_prevalence:.4f} "
      f"({int(y['test'].sum()):,} illicit / {len(y['test']):,} labeled)")

check("baseline: constant score AP equals the evaluation prevalence",
      round(baseline_constant_ap, 6), round(test_prevalence, 6),
      note="AP of a constant score is the positive rate of the evaluated window, not of the training window")

## 7. Logistic Regression

Logistic Regression answers: *how much predictive power is in these features without tree
interactions?*

It is scale-sensitive, so the preprocessing is a `Pipeline`: median imputation, then
`StandardScaler`. The pipeline is fitted on the fit window (1–24) only and applied unchanged to
validation and test — the scaler never sees a validation or test row, which is the classic leak
this step exists to prevent. (The 165 model features have no missing values, so the imputer is a
no-op here; it is kept so the pipeline stays correct if a feature set with missingness is swapped
in.)

`class_weight="balanced"` handles the 1:7.6 imbalance in the fit window. The reported operating
point is not the default 0.5: the threshold is chosen on the validation window by maximising F1,
frozen, then applied to the test period.

In [ ]:
def build_logistic_model() -> Pipeline:
    """Impute, scale and classify. Fitted only on training-window rows."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2_000,
                                     solver="lbfgs", random_state=SEED)),
    ])


logistic_selection = build_logistic_model()
started = time.time()
logistic_selection.fit(X["fit"], y["fit"])
logistic_fit_seconds = time.time() - started
print(f"logistic regression fitted on 1-24 in {logistic_fit_seconds:.1f}s")

logistic_validation_scores = logistic_selection.predict_proba(X["validation"])[:, 1]
print(f"validation PR-AUC (selection model): "
      f"{average_precision_score(y['validation'], logistic_validation_scores):.4f}")

logistic_final = build_logistic_model()
logistic_final.fit(X["train"], y["train"])
print("logistic regression refit on 1-34")

joblib.dump(logistic_final, OUT_DIR / "logistic_regression.joblib")

## 8. XGBoost

The first real model. Choices, and why:

* **No feature scaling, no `log1p`.** Trees split on order, so monotone rescaling changes nothing.
  The EDA found skewness up to 148; a tree does not care, and a transform that is not measured is
  not added. Section 12 lets the measured feature importance show what the model actually uses.
* **`scale_pos_weight = negatives / positives`, computed on the training window only.** It changes
  the gradient weighting, never the data. No resampling, no synthetic rows, and the test
  distribution is left exactly as the world produced it.
* **Conservative first pass.** `max_depth=6`, `learning_rate=0.05`, `min_child_weight=5`,
  `subsample=0.8`, `colsample_bytree=0.8`, `reg_lambda=1.0`, `tree_method="hist"` — CPU-friendly
  and small enough to attribute failures honestly.
* **`eval_metric="aucpr"`** so early stopping optimizes the primary metric directly.
* **Early stopping on the validation window**, up to 500 trees with a 50-round patience. The tree
  count that survives is then reused for a refit on the full 1–34 training period. The test period
  is not passed to `fit` and not used for either decision.

In [ ]:
scale_pos_weight_fit = float((len(y["fit"]) - y["fit"].sum()) / y["fit"].sum())
scale_pos_weight_train = float((len(y["train"]) - y["train"].sum()) / y["train"].sum())

XGB_PARAMS = {
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "random_state": SEED,
    "n_jobs": 4,
}
MAX_TREES = 500
PATIENCE = 50
def make_xgb(scale_pos_weight: float, n_estimators: int, patience):
    """Build an XGBClassifier, putting early stopping wherever this xgboost version accepts it.

    The sklearn wrapper has moved ``early_stopping_rounds`` between the constructor and ``fit``
    across releases, and it forwards unknown constructor keywords through ``**kwargs`` instead of
    raising, so probing the signature is unreliable. Try the constructor, fall back to ``fit``,
    and say which one was used. Returns ``(estimator, patience_belongs_to_fit)``.
    """
    params = dict(XGB_PARAMS, n_estimators=n_estimators, scale_pos_weight=scale_pos_weight)
    if patience is None:
        return xgb.XGBClassifier(**params), False
    try:
        return xgb.XGBClassifier(**params, early_stopping_rounds=patience), False
    except TypeError:
        return xgb.XGBClassifier(**params), True


xgb_selection, patience_at_fit = make_xgb(scale_pos_weight_fit, MAX_TREES, PATIENCE)
fit_kwargs = {"eval_set": [(X["validation"], y["validation"])], "verbose": False}
if patience_at_fit:
    fit_kwargs["early_stopping_rounds"] = PATIENCE
print(f"early stopping passed to {'fit()' if patience_at_fit else 'the constructor'} "
      f"(xgboost {xgb.__version__})")

started = time.time()
xgb_selection.fit(X["fit"], y["fit"], **fit_kwargs)
xgb_fit_seconds = time.time() - started

best_iteration = int(getattr(xgb_selection, "best_iteration", 0) or 0)
if best_iteration <= 0:
    best_iteration = int(xgb_selection.get_booster().best_iteration)
selected_trees = max(1, best_iteration + 1)

print(f"selection model: {selected_trees} trees kept of {MAX_TREES} (early stopping, patience {PATIENCE})")
print(f"fit time         : {xgb_fit_seconds:.1f}s")
print(f"validation PR-AUC: {average_precision_score(y['validation'], xgb_selection.predict_proba(X['validation'])[:, 1]):.4f}")
print(f"scale_pos_weight : {scale_pos_weight_fit:.3f} (fit) / {scale_pos_weight_train:.3f} (refit)")

check("xgboost: early stopping kept at least one tree", selected_trees >= 1, True)
check("xgboost: early stopping stopped before the tree cap", selected_trees < MAX_TREES, True,
      note="a model that used all 500 trees hit the cap and should be re-run with more")

In [ ]:
xgb_final = make_xgb(scale_pos_weight_train, selected_trees, None)[0]

started = time.time()
xgb_final.fit(X["train"], y["train"], verbose=False)
xgb_refit_seconds = time.time() - started
print(f"final model refit on 1-34: {selected_trees} trees in {xgb_refit_seconds:.1f}s")

xgb_final.get_booster().save_model(str(OUT_DIR / "xgb_model.json"))
print(f"saved {OUT_DIR / 'xgb_model.json'}")

xgb_hyperparameters = {
    **XGB_PARAMS,
    "n_estimators": selected_trees,
    "early_stopping_rounds": PATIENCE,
    "max_trees_tried": MAX_TREES,
    "scale_pos_weight_fit": scale_pos_weight_fit,
    "scale_pos_weight_refit": scale_pos_weight_train,
    "fit_window": "1-24",
    "validation_window": "25-34",
    "refit_window": "1-34",
    "features": len(model_features),
    "fit_seconds_selection": round(xgb_fit_seconds, 1),
    "refit_seconds": round(xgb_refit_seconds, 1),
    "seed": SEED,
    "xgboost_version": xgb.__version__,
}
save_json("xgb_hyperparameters.json", xgb_hyperparameters)
display(pd.Series(xgb_hyperparameters, name="value").to_frame())

## 9. Model Evaluation

Primary metric: **PR-AUC (average precision)**. With 6.50% prevalence in the test period, accuracy
is uninformative — a model that never predicts fraud scores 93.5% accuracy and is worthless.

Threshold-independent metrics first, on the untouched test period:

* PR-AUC / average precision — the primary number.
* ROC-AUC — reported for comparison, but it flatters rare-class models.

Then the operating point. **The threshold is chosen on the validation window (25–34) by maximising
F1, frozen, and applied to the test period. It is never tuned against test results or test
prevalence.** Section 14 reports a second, training-prevalence threshold as a sensitivity check,
because the validation window's illicit rate (19.77%) is far higher than the test period's (6.50%).

Both error types are reported because the costs are asymmetric and opposite:

* **False negatives** are undetected illicit transactions. In this domain that is money laundered
  and a compliance failure.
* **False positives** are licit transactions flagged for review. That is analyst time, and at scale
  it is the reason fraud teams stop trusting a model.

Neither is arbitrarily optimised. The PR curve and the confusion matrix are reported so the tradeoff
is visible and can be chosen by whoever owns the cost function, not by the model.

In [ ]:
def evaluate(y_true, scores, threshold: float) -> dict:
    """All reported metrics for one model on one window, at one fixed threshold."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype="float64")
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    prevalence = float(y_true.mean())
    pr_auc = float(average_precision_score(y_true, scores))
    return {
        "n": int(y_true.size),
        "illicit": int(y_true.sum()),
        "prevalence": prevalence,
        "threshold": float(threshold),
        "pr_auc": pr_auc,
        "pr_auc_lift_over_prevalence": pr_auc / prevalence if prevalence else float("nan"),
        "roc_auc": float(roc_auc_score(y_true, scores)) if 0 < y_true.sum() < y_true.size else float("nan"),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "predicted_positives": int(predicted.sum()),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }


def select_threshold(y_true, scores) -> tuple:
    """F1-maximising threshold on validation data only."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype="float64")
    best_threshold, best_f1 = 0.5, -1.0
    for candidate in np.round(np.linspace(0.005, 0.995, 199), 4):
        value = f1_score(y_true, (scores >= candidate).astype(int), zero_division=0)
        if value > best_f1:
            best_threshold, best_f1 = float(candidate), float(value)
    return best_threshold, best_f1


logistic_scores = {"validation": logistic_selection.predict_proba(X["validation"])[:, 1],
                   "test": logistic_final.predict_proba(X["test"])[:, 1]}
xgb_scores = {"validation": xgb_selection.predict_proba(X["validation"])[:, 1],
              "test": xgb_final.predict_proba(X["test"])[:, 1]}

operating_points = {}
for name, scores in (("logistic_regression", logistic_scores), ("xgboost", xgb_scores)):
    threshold, validation_f1 = select_threshold(y["validation"], scores["validation"])
    operating_points[name] = {"threshold": threshold, "validation_f1": validation_f1}
    print(f"{name:<21} threshold {threshold:.3f} (validation F1 {validation_f1:.4f})")

prevalence_threshold = baseline_prevalence_train
print(f"secondary sensitivity threshold (train prevalence): {prevalence_threshold:.4f}")

In [ ]:
test_results = {
    "logistic_regression": evaluate(y["test"], logistic_scores["test"], operating_points["logistic_regression"]["threshold"]),
    "xgboost": evaluate(y["test"], xgb_scores["test"], operating_points["xgboost"]["threshold"]),
}

for name, result in test_results.items():
    print(f"=== {name} — test period 35-49 ===")
    print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in result.items()}, indent=2))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for name, scores in (("logistic_regression", logistic_scores), ("xgboost", xgb_scores)):
    precision, recall, _ = precision_recall_curve(y["test"], scores["test"])
    axes[0].plot(recall, precision, label=f"{name} (AP={test_results[name]['pr_auc']:.3f})")
axes[0].axhline(test_prevalence, color="grey", linestyle="--", label=f"prevalence {test_prevalence:.3f}")
axes[0].set(xlabel="recall", ylabel="precision", title="Precision-recall curve (test 35-49)")
axes[0].legend(loc="upper right", fontsize=8)

for name, scores in (("logistic_regression", logistic_scores), ("xgboost", xgb_scores)):
    axes[1].hist(scores["test"][y["test"].to_numpy() == 0], bins=60, alpha=0.5, label=f"{name} licit")
    axes[1].hist(scores["test"][y["test"].to_numpy() == 1], bins=60, alpha=0.5, label=f"{name} illicit")
axes[1].set(xlabel="predicted score", ylabel="count", title="Score distribution (test 35-49)")
axes[1].set_yscale("log")
axes[1].legend(fontsize=7)
save_fig("test_pr_curve_and_scores.png")

In [ ]:
comparison_rows = []
for _, row in baseline_table.iterrows():
    comparison_rows.append({"model": row["model"], "pr_auc": row["pr_auc"], "roc_auc": row["roc_auc"],
                            "precision": row["precision"], "recall": row["recall"], "f1": row["f1"],
                            "threshold": row["threshold"]})
for name, label in (("logistic_regression", "Logistic Regression"), ("xgboost", "XGBoost")):
    comparison_rows.append({"model": label, **{k: test_results[name][k] for k in
                                               ("pr_auc", "roc_auc", "precision", "recall", "f1", "threshold")}})
model_comparison = pd.DataFrame(comparison_rows)
save_table("model_comparison.csv", model_comparison)
display(model_comparison.round(4))

metrics_payload = {
    "protocol": {"fit": "1-24", "validation": "25-34", "train": "1-34", "test": "35-49",
                 "test_early": "35-42", "test_drift": "43-49"},
    "features": {"n": len(model_features), "list": model_features},
    "operating_points": operating_points,
    "prevalence_threshold_sensitivity": prevalence_threshold,
    "test": {"baseline_prevalence": test_prevalence,
             "logistic_regression": test_results["logistic_regression"],
             "xgboost": test_results["xgboost"]},
    "hyperparameters": {"xgboost": xgb_hyperparameters,
                        "logistic_regression": {"class_weight": "balanced", "max_iter": 2_000,
                                                "scaler": "StandardScaler on the fit window only"},
                        "fit_seconds": {"logistic_regression": round(logistic_fit_seconds, 1),
                                        "xgboost_selection": round(xgb_fit_seconds, 1),
                                        "xgboost_refit": round(xgb_refit_seconds, 1)}},
}
save_json("metrics.json", metrics_payload)
print(f"saved {OUT_DIR / 'metrics.json'}")

## 10. Temporal Evaluation

The aggregate test number can hide a lot. The test period is split into two non-overlapping
windows and each is scored with the frozen threshold and the frozen model:

| Window | Steps | Why |
| :--- | :--- | :--- |
| `35–42` | 8 steps | earlier part of the test period |
| `43–49` | 7 steps | the recent low-prevalence regime: 169 illicit of 6,687 labeled (2.53%) |
| `35–49` | 15 steps | the headline test result |

`43–49` was rejected as the *primary* test in the EDA because 169 positives give a noisy estimate,
but it is exactly the regime a deployed model would meet, so it is reported.

**This is not drift monitoring.** No change between `35–42` and `43–49` is labelled "drift" here —
with one split there is nothing to compare a single shift against. What is measured is temporal
generalization: does a model fitted on 1–34 still rank transactions correctly when the illicit rate
drops by a factor of four? Formal drift detection is a later phase.

In [ ]:
# Masks are relative to the test rows, so they index the test-length score arrays directly.
test_step_array = test_step.to_numpy()
y_test_array_full = y["test"].to_numpy().astype(int)
test_windows = {
    "test": ("35-49 (primary)", np.ones(len(test_step_array), dtype=bool)),
    "test_early": ("35-42", test_step_array <= TEST_EARLY_MAX),
    "test_drift": ("43-49", test_step_array >= DRIFT_MIN),
}

temporal_rows = []
for key, (label, mask) in test_windows.items():
    y_window = y_test_array_full[mask]
    row = {"window": label, "steps": {"test": "35-49", "test_early": "35-42",
                                      "test_drift": "43-49"}[key],
           "labeled": int(len(y_window)), "illicit": int(y_window.sum()),
           "prevalence": float(y_window.mean())}
    for name, scores in (("logistic_regression", logistic_scores), ("xgboost", xgb_scores)):
        result = evaluate(y_window, scores["test"][mask], operating_points[name]["threshold"])
        row[f"{name}_pr_auc"] = result["pr_auc"]
        row[f"{name}_roc_auc"] = result["roc_auc"]
        row[f"{name}_precision"] = result["precision"]
        row[f"{name}_recall"] = result["recall"]
        row[f"{name}_f1"] = result["f1"]
        row[f"{name}_tp"] = result["tp"]
        row[f"{name}_fp"] = result["fp"]
        row[f"{name}_fn"] = result["fn"]
    temporal_rows.append(row)

temporal_table = pd.DataFrame(temporal_rows)
save_table("temporal_metrics.csv", temporal_table)
display(temporal_table.round(4))

figure, axis = plt.subplots(figsize=(8, 4.4))
positions = np.arange(len(temporal_table))
width = 0.35
axis.bar(positions - width / 2, temporal_table["logistic_regression_pr_auc"], width,
         label="Logistic Regression")
axis.bar(positions + width / 2, temporal_table["xgboost_pr_auc"], width, label="XGBoost")
for position, value in zip(positions, temporal_table["prevalence"]):
    axis.plot([position - 0.4, position + 0.4], [value, value], color="grey",
              linestyle="--", linewidth=1)
axis.set_xticks(positions, temporal_table["steps"])
axis.set(xlabel="test sub-window", ylabel="PR-AUC", title="PR-AUC by temporal window (dashed line = prevalence)")
axis.legend()
save_fig("temporal_pr_auc.png")

check("temporal: sub-windows partition the test set",
      int(temporal_table.loc[temporal_table['steps'] != '35-49', 'labeled'].sum()),
      int(temporal_table.loc[temporal_table['steps'] == '35-49', 'labeled'].iloc[0]))

## 11. Error Analysis

Aggregate metrics say how many errors there are; this section says what they look like. Only
aggregates and masked examples are printed — no raw identifiers, and nothing that would expose a
specific address.

Four questions:

1. **Where do errors sit in time?** Error counts per step show whether the model degrades inside the
   test period rather than only between train and test.
2. **Do false positives and false negatives have different scores?** Barely-over-threshold false
   positives are cheap to discount; high-confidence false negatives are the dangerous ones.
3. **Is the address-less population different?** 965 transactions have no address links. If they are
   systematically mispredicted, `has_addresses` is doing what it was added for.
4. **What does the confusion matrix cost?** Reported as counts, with the asymmetric-cost reading
   made explicit rather than resolved by the notebook.

In [ ]:
def mask_tx_id(value: str) -> str:
    """Shorten an identifier for display; the full value stays in predictions.csv."""
    return f"{value[:6]}...{value[-4:]}" if len(value) > 12 else "***"


xgb_test_scores = xgb_scores["test"]
xgb_threshold = operating_points["xgboost"]["threshold"]
xgb_predicted = (xgb_test_scores >= xgb_threshold).astype(int)
y_test_array = y["test"].to_numpy().astype(int)

error_frame = pd.DataFrame({
    "time_step": test_step,
    "y_true": y_test_array,
    "score": xgb_test_scores,
    "predicted": xgb_predicted,
    "has_addresses": X["test"]["has_addresses"].astype(int).to_numpy(),
})
error_frame["error_type"] = np.select(
    [(error_frame.y_true == 1) & (error_frame.predicted == 0),
     (error_frame.y_true == 0) & (error_frame.predicted == 1)],
    ["FN", "FP"], default="correct")

def error_counts(block: pd.DataFrame) -> pd.Series:
    """tp/fp/fn/tn plus precision, recall and the positive rate, for one slice of the test set."""
    positive, negative = block.y_true == 1, block.y_true == 0
    flagged = block.predicted == 1
    tp, fp = int((positive & flagged).sum()), int((negative & flagged).sum())
    fn, tn = int((positive & ~flagged).sum()), int((negative & ~flagged).sum())
    return pd.Series({
        "n": len(block),
        "illicit": int(positive.sum()),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": tp / (tp + fp) if tp + fp else float("nan"),
        "recall": tp / (tp + fn) if tp + fn else float("nan"),
        "flagged_rate": float(flagged.mean()),
    })


per_step = (error_frame.groupby("time_step", group_keys=False)
            .apply(lambda block: error_counts(block), include_groups=False)
            .reset_index())
save_table("errors_by_step.csv", per_step)
display(per_step.round(3))

subgroup_rows = []
for value, label in ((1, "has address link"), (0, "no address link")):
    row = {"group": label, **error_counts(error_frame[error_frame.has_addresses == value]).to_dict()}
    subgroup_rows.append(row)
subgroup = pd.DataFrame(subgroup_rows).set_index("group")
save_table("errors_by_address_linkage.csv", subgroup.reset_index())
display(subgroup.round(4))

error_summary = {
    "fn": int((error_frame.error_type == "FN").sum()),
    "fp": int((error_frame.error_type == "FP").sum()),
    "fn_median_score": float(error_frame.loc[error_frame.error_type == "FN", "score"].median()),
    "fp_median_score": float(error_frame.loc[error_frame.error_type == "FP", "score"].median()),
    "threshold": xgb_threshold,
    "example_false_negatives": [mask_tx_id(v) for v in test_id[error_frame.error_type.to_numpy() == "FN"][:5]],
    "example_false_positives": [mask_tx_id(v) for v in test_id[error_frame.error_type.to_numpy() == "FP"][:5]],
}
print(json.dumps(error_summary, indent=2))
save_json("error_analysis.json", error_summary)

check("errors: tp+fp+fn+tn covers every test row",
      int((error_frame.predicted == error_frame.y_true).sum()
          + (error_frame.error_type == "FN").sum() + (error_frame.error_type == "FP").sum()),
      len(error_frame))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].plot(per_step["time_step"], per_step["fn"], marker="o", label="false negatives")
axes[0].plot(per_step["time_step"], per_step["fp"], marker="s", label="false positives")
axes[0].set(xlabel="time step", ylabel="count", title="Errors per time step (test 35-49)")
axes[0].legend()

for label, mask in (("illicit", y_test_array == 1), ("licit", y_test_array == 0)):
    axes[1].hist(error_frame.loc[mask, "score"], bins=60, alpha=0.6, label=label)
axes[1].axvline(xgb_threshold, color="black", linestyle="--", label=f"threshold {xgb_threshold:.3f}")
axes[1].set(xlabel="predicted score", ylabel="count", title="XGBoost scores on the test period", yscale="log")
axes[1].legend()
save_fig("error_analysis.png")

## 12. Feature Importance

Two views, because they disagree in informative ways:

* **Gain** — the average loss reduction a split on the feature buys, summed over the trees. Measures
  what the fitted model used internally, and is biased toward high-cardinality continuous columns.
* **Permutation importance** — the drop in validation PR-AUC when a feature is shuffled. Measures
  what the model needs for its *current* performance, and is measured on the validation window, so
  it costs nothing from the test period.

Importance is **not** causality. A feature can rank highly because it correlates with a real cause,
because it splits a convenient subpopulation, or because another feature redundantly covers it.
The section's job is to check consistency with the EDA, not to explain illicit behaviour.

In [ ]:
booster = xgb_final.get_booster()
gain = booster.get_score(importance_type="gain")
gain_table = (pd.DataFrame({"feature": list(gain.keys()), "gain": list(gain.values())})
              .sort_values("gain", ascending=False)
              .reset_index(drop=True))

sample_size = min(4_000, len(X["validation"]))
permutation = permutation_importance(
    xgb_final, X["validation"].head(sample_size), y["validation"].head(sample_size),
    scoring="average_precision", n_repeats=5, random_state=SEED, n_jobs=4,
)
permutation_table = (pd.DataFrame({"feature": X["validation"].columns,
                                   "permutation_importance": permutation.importances_mean})
                     .sort_values("permutation_importance", ascending=False)
                     .reset_index(drop=True))

importance_table = (gain_table.merge(permutation_table, on="feature", how="outer")
                    .sort_values("gain", ascending=False)
                    .reset_index(drop=True))
importance_table["gain_rank"] = importance_table["gain"].rank(ascending=False)
importance_table["permutation_rank"] = importance_table["permutation_importance"].rank(ascending=False)
save_table("feature_importance.csv", importance_table)
display(importance_table.head(20).round(4))

print("top 10 by gain             :", ", ".join(gain_table["feature"].head(10)))
print("top 10 by permutation      :", ", ".join(permutation_table["feature"].head(10)))

figure, axes = plt.subplots(1, 2, figsize=(13, 5.4))
axes[0].barh(gain_table["feature"].head(15)[::-1], gain_table["gain"].head(15)[::-1])
axes[0].set(title="XGBoost gain, top 15", xlabel="gain")
axes[1].barh(permutation_table["feature"].head(15)[::-1],
             permutation_table["permutation_importance"].head(15)[::-1])
axes[1].set(title="Permutation importance (validation PR-AUC drop), top 15", xlabel="mean drop")
save_fig("feature_importance.png")

top_gain = gain_table["feature"].head(5).tolist()
print("\nCross-check against the EDA (docs/EDA.md, Feature Findings):")
print("EDA: strongest point-biserial correlation was Local_feature_53 at r = -0.26, signal weak and diffuse.")
print(f"Model: top 5 by gain are {top_gain}.")

## 13. Model Comparison

All rows are measured on the identical test period (35–49), with the identical metric
implementation, and each learned model at its own validation-chosen threshold. The baseline rows are
shown for scale. No cell in this table is estimated or filled in.

In [ ]:
display(model_comparison.round(4))

xgb_result = test_results["xgboost"]
logistic_result = test_results["logistic_regression"]
digest = f"""
BitcoinGraphGuard - Phase 2 XGBoost baseline digest ({time.strftime('%Y-%m-%d %H:%M')})
----------------------------------------------------------------------------------
features        : {len(model_features)} (165 Local/Aggregate + has_addresses), 17 domain columns excluded
population      : class 1 + class 2 only; class 3 (unknown) never used
protocol        : fit 1-24 | validation 25-34 | train(refit) 1-34 | test 35-49 (frozen)
test prevalence : {test_prevalence:.4f} ({int(y['test'].sum()):,} illicit / {len(y['test']):,} labeled)
operating point : validation F1-max, frozen -> xgb {operating_points['xgboost']['threshold']:.3f}, lr {operating_points['logistic_regression']['threshold']:.3f}
prevalence base : PR-AUC {baseline_constant_ap:.4f} (= test prevalence; a constant score is scored against the window it is evaluated on), ROC-AUC 0.500
logistic reg    : PR-AUC {logistic_result['pr_auc']:.4f} | ROC-AUC {logistic_result['roc_auc']:.4f} | P {logistic_result['precision']:.4f} | R {logistic_result['recall']:.4f} | F1 {logistic_result['f1']:.4f} | TP {logistic_result['tp']} FP {logistic_result['fp']} FN {logistic_result['fn']}
xgboost         : PR-AUC {xgb_result['pr_auc']:.4f} | ROC-AUC {xgb_result['roc_auc']:.4f} | P {xgb_result['precision']:.4f} | R {xgb_result['recall']:.4f} | F1 {xgb_result['f1']:.4f} | TP {xgb_result['tp']} FP {xgb_result['fp']} FN {xgb_result['fn']}
xgboost trees   : {selected_trees} (early stopping, cap {MAX_TREES}), fit {xgb_fit_seconds:.0f}s + refit {xgb_refit_seconds:.0f}s
temporal        : {', '.join(f"{row.steps} PR-AUC {row.xgboost_pr_auc:.4f} (prev {row.prevalence:.4f})" for row in temporal_table.itertuples())}
errors          : FN {error_summary['fn']} (median score {error_summary['fn_median_score']:.3f}) | FP {error_summary['fp']} (median score {error_summary['fp_median_score']:.3f})
address-less    : {int(subgroup.loc['no address link', 'n'])} test rows, recall {subgroup.loc['no address link', 'recall']:.3f} vs {subgroup.loc['has address link', 'recall']:.3f} for linked
"""
print(digest)
(OUT_DIR / "xgboost_digest.txt").write_text(digest, encoding="utf-8")

predictions = pd.DataFrame({
    "txId": test_id,
    "time_step": test_step,
    "y_true": y_test_array,
    "logistic_score": logistic_scores["test"],
    "logistic_predicted": (logistic_scores["test"] >= operating_points["logistic_regression"]["threshold"]).astype(int),
    "xgboost_score": xgb_test_scores,
    "xgboost_predicted": xgb_predicted,
    "has_addresses": X["test"]["has_addresses"].astype(int).to_numpy(),
})
save_table("predictions.csv", predictions)
save_table("checks.csv", pd.DataFrame(CHECKS))

failures = [row for row in CHECKS if not row["ok"]]
print(f"{len(CHECKS)} checks, {len(failures)} failing")
if failures:
    display(pd.DataFrame(failures))
print("artifacts:", sorted(p.name for p in OUT_DIR.iterdir()))

## 14. Conclusions

**This section is written after the notebook has been run.** The digest cell in section 13 prints
the measured values and writes `xgboost_digest.txt`; `docs/XGBOOST.md` is populated from that
output, not from expectation.

What has to be answered from the numbers above:

1. **Does XGBoost beat the prevalence headline?** Compare `pr_auc` against the prevalence baseline
   row. The ratio `pr_auc_lift_over_prevalence` in `metrics.json` states it directly.
2. **Does XGBoost beat Logistic Regression?** If the trees add little over a linear model on the
   same 166 columns, the remaining headroom is in graph structure, not in nonlinearity — which is
   the argument Phase 3 depends on.
3. **Does performance hold in `43–49`?** The illicit rate there is 2.53% against 11.58% in training.
   A large PR-AUC drop is a generalization finding to report, not drift to name.
4. **Which error dominates, and at what confidence?** The median score of false negatives against
   the operating threshold says whether failures are near-misses or confident mistakes.
5. **Do the important features agree with the EDA?** Agreement between the gain ranking and the
   point-biserial correlations in `docs/EDA.md` is evidence the model learned the same signal the
   EDA measured.
6. **Do the 965 address-less transactions behave differently?** Compare their recall with the linked
   population in `errors_by_address_linkage.csv`.

### What this baseline cannot capture

It scores each transaction in isolation. Elliptic++'s own EDA result — transaction components are
confined to a single time step, and cross-step signal travels through address nodes — is precisely
what a per-row model cannot use. Money-flow chains, shared addresses between a licit and an illicit
transaction, and multi-hop structure around a node are all invisible here.

That is the gap Phase 3 has to close. The PR-AUC measured above is the number GraphSAGE and the
heterogeneous models must beat under the same protocol; if they do not, the added complexity is not
paying for itself.

### Threshold caveat to carry forward

The threshold was selected on a validation window with a 19.77% illicit rate and applied to a test
period with a 6.50% rate. Operating points do not transfer across prevalence shifts unchanged, so
any serving decision must re-derive the threshold on data that matches the deployment regime — and
never on the test set.